<a href="https://colab.research.google.com/github/goutham3010/Hospital-Patient-Flow-and-Resource-Demand-Prediction-System/blob/main/07_Explainable_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL PATIENT FLOW & DEMAND PREDICTION SYSTEM
## NOTEBOOK 07: EXPLAINABLE AI (TREESHAP)
**Workflow Phase 10** of the 17-Stage Hospital Operations Decision Support Architecture.

### Objectives:
1. Deconstruct black-box machine learning predictions into additive Shapley contributions.
2. Identify the **Global Feature Importance** hierarchy across the 10-year observational period.
3. Produce instance-level **Local Explanations** answering: *Why did the model predict an influx surge on this specific date?*
4. Map raw mathematical feature names to human-readable clinical operations terminology.


In [ ]:
# 1. IMPORT LIBRARIES & EXPLAINABILITY ENGINE
import pandas as pd
import numpy as np
import shap
import joblib
import matplotlib.pyplot as plt
import sys

sys.path.append('src')
from explainability import explain_single_prediction, get_feature_human_label

print("SHAP modules loaded.")


In [ ]:
# 2. LOAD TRAINED MODEL AND TEST HORIZON
bundle = joblib.load('models/patient_flow_model.pkl')
model = bundle['final_model']
feature_cols = bundle['feature_cols']

df = pd.read_csv('data/daily_patient_flow.csv')
test_df = df[df['date'] >= '2024-01-01'].reset_index(drop=True)
X_test = test_df[feature_cols]

print(f"Loaded model and {len(X_test)} evaluation instances.")


In [ ]:
# 3. COMPUTE TREESHAP VALUES
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print("TreeSHAP values computed successfully.")
print(f"Expected Base Value: {float(explainer.expected_value):.2f} patients")


In [ ]:
# 4. GLOBAL FEATURE IMPORTANCE SUMMARY PLOT
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, max_display=12, show=False)
plt.title("TreeSHAP Global Feature Importance Ranking (Patient Demand Drivers)", fontsize=13, fontweight='bold')
plt.show()


In [ ]:
# 5. LOCAL INSTANCE EXPLANATION (SURGE DAY BREAKDOWN)
# We inspect sample index 45 (an acute surge date)
sample_idx = 45
sample_row = test_df.iloc[sample_idx]

exp = explain_single_prediction(model, explainer, sample_row, feature_cols, top_k=4)

print(f"Date: {sample_row['date']}")
print(f"Summary: {exp['summary_text']}\n")

print("=== TOP SURGE ACCELERATORS (+ Increases Demand) ===")
for p in exp['positive_drivers']:
    print(f" [+] {p['label']:<35} : +{p['shap_value']:.2f} patients (Value: {p['actual_value']})")

print("\n=== TOP DAMPENING FACTORS (- Decreases Demand) ===")
for n in exp['negative_drivers']:
    print(f" [-] {n['label']:<35} : {n['shap_value']:.2f} patients (Value: {n['actual_value']})")
